# 🏋️ QUY TRÌNH HUẤN LUYỆN MÔ HÌNH MODERNBERT NLI
Notebook này tập trung hoàn toàn vào việc fine-tune mô hình ModernBERT trên tập dữ liệu mâu thuẫn văn bản pháp lý.


In [ ]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer, util

# ======== Thiết bị ========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ======== Model paraphrase để match câu hỏi ========
embed_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embed_model = SentenceTransformer(embed_model_name, device=device)

# ======== Model NLI để kiểm tra mâu thuẫn câu trả lời ========
nli_model_name = "facebook/bart-large-mnli"
nli_pipe = pipeline("text-classification", model=nli_model_name, device=0 if torch.cuda.is_available() else -1)




In [ ]:
from datasets import load_dataset

def load_dnli_examples(n=10):
    # Tải dataset DNLI (train split)
    dataset = load_dataset('dialogue_nli/dnli', split='train')
    
    # Lấy n ví dụ đầu tiên
    examples = []
    for i in range(min(n, len(dataset))):
        item = dataset[i]
        examples.append({
            "premise": item["premise"],          # câu gốc / turn 1
            "hypothesis": item["hypothesis"],    # câu tiếp theo / turn 2
            "label": item["label"]               # 0=entailment, 1=neutral, 2=contradiction
        })
    return examples

# Sử dụng hàm
examples = load_dnli_examples(10)
for ex in examples:
    print("Premise:", ex["premise"])
    print("Hypothesis:", ex["hypothesis"])
    print("Label:", ex["label"])
    print("-----")


## 2. Khởi tạo Mô hình & Tokenizer
Sử dụng kiến trúc ModernBERT làm nền tảng cho việc fine-tuning.


In [ ]:
!pip install transformers datasets evaluate huggingface_hub accelerate -q

import json
import logging
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForQuestionAnswering, 
    TrainingArguments, 
    Trainer
)
import evaluate
import torch
import numpy as np
from typing import Dict, List, Any

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# -----------------------------
# 1. Load JSONL dataset
# -----------------------------
def load_jsonl_dataset(file_path: str) -> List[Dict]:
    """Load dataset from JSONL file"""
    try:
        raw_data = []
        with open(file_path, "r", encoding="utf-8") as f:
            for line_num, line in enumerate(f, 1):
                if line.strip():  # Skip empty lines
                    try:
                        raw_data.append(json.loads(line))
                    except json.JSONDecodeError as e:
                        logger.warning(f"Skipping invalid JSON on line {line_num}: {e}")
        
        logger.info(f"Loaded {len(raw_data)} samples from {file_path}")
        return raw_data
    except Exception as e:
        logger.error(f"Error loading dataset: {e}")
        raise

# Load your dataset
file_path = "/kaggle/input/datasetlegal1/merged_legal_dataset.jsonl"
raw_data = load_jsonl_dataset(file_path)

# -----------------------------
# 2. Explore and fix dataset structure
# -----------------------------
def explore_dataset_structure(data: List[Dict]) -> None:
    """Explore dataset structure"""
    if len(data) > 0:
        logger.info(f"Dataset size: {len(data)}")
        sample = data[0]
        logger.info("Sample keys:", list(sample.keys()))
        logger.info("Sample structure:")
        for key, value in sample.items():
            if isinstance(value, list):
                logger.info(f"  {key}: list with {len(value)} items")
                if len(value) > 0:
                    logger.info(f"    First item: {value[0]}")
            elif isinstance(value, str):
                logger.info(f"  {key}: '{value[:100]}{'...' if len(value) > 100 else ''}'")
            else:
                logger.info(f"  {key}: {type(value)} - {value}")

explore_dataset_structure(raw_data)

# -----------------------------
# 3. Convert to QA format with proper error handling
# -----------------------------
def convert_to_qa(example: Dict) -> Dict:
    """Convert legal data to QA format"""
    question = "Find span in premise that contradicts hypothesis"
    context = example.get("premise", "").strip()
    
    # Get highlight_inner - handle different formats
    highlight_inner = example.get("highlight_inner", [])
    
    # Ensure highlight_inner is a list
    if isinstance(highlight_inner, str):
        highlight_inner = [highlight_inner]
    elif not isinstance(highlight_inner, list):
        highlight_inner = []
    
    # Clean empty spans
    highlight_inner = [span.strip() for span in highlight_inner if span and span.strip()]
    
    if not highlight_inner:
        # No valid highlights - create dummy answer
        return {
            "question": question,
            "context": context,
            "answers": {
                "text": [""],
                "answer_start": [0]
            }
        }
    
    # Find positions of all spans
    answer_texts = []
    answer_starts = []
    
    for span in highlight_inner:
        start_idx = context.find(span)
        if start_idx != -1:
            answer_texts.append(span)
            answer_starts.append(start_idx)
        else:
            # Try fuzzy matching for spans with different whitespace
            clean_span = ' '.join(span.split())
            clean_context = ' '.join(context.split())
            fuzzy_start = clean_context.find(clean_span)
            
            if fuzzy_start != -1:
                # Map back to original context
                words_before = len(clean_context[:fuzzy_start].split())
                original_words = context.split()
                if words_before < len(original_words):
                    original_start = len(' '.join(original_words[:words_before]))
                    if words_before > 0:
                        original_start += 1  # account for space
                    answer_texts.append(span)
                    answer_starts.append(original_start)
                else:
                    logger.warning(f"Could not find span '{span[:50]}...' in context")
            else:
                logger.warning(f"Could not find span '{span[:50]}...' in context")
    
    # If no spans found, use dummy
    if not answer_texts:
        answer_texts = [""]
        answer_starts = [0]
    
    return {
        "question": question,
        "context": context,
        "answers": {
            "text": answer_texts,
            "answer_start": answer_starts
        }
    }

# Convert data and filter valid samples
qa_data = []
for i, example in enumerate(raw_data):
    try:
        qa_example = convert_to_qa(example)
        if qa_example["context"]:  # Only keep examples with non-empty context
            qa_data.append(qa_example)
    except Exception as e:
        logger.warning(f"Error processing example {i}: {e}")

logger.info(f"Converted {len(qa_data)} valid QA examples")

# Show sample
if qa_data:
    sample = qa_data[0]
    logger.info("Sample QA example:")
    logger.info(f"Question: {sample['question']}")
    logger.info(f"Context: {sample['context'][:200]}...")
    logger.info(f"Answers: {sample['answers']}")

# -----------------------------
# 4. Split dataset
# -----------------------------
if len(qa_data) < 10:
    raise ValueError("Dataset too small for training. Need at least 10 samples.")

split_idx = int(0.8 * len(qa_data))
train_data = qa_data[:split_idx]
test_data = qa_data[split_idx:]

train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

logger.info(f"Train samples: {len(train_dataset)}")
logger.info(f"Test samples: {len(test_dataset)}")

# -----------------------------
# 5. Load model and tokenizer
# -----------------------------
model_name = "deepset/roberta-base-squad2"  # Good choice for NLI tasks
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForQuestionAnswering.from_pretrained(model_name)
    logger.info(f"Loaded model: {model_name}")
except Exception as e:
    logger.error(f"Error loading model: {e}")
    # Fallback to a more common model
    model_name = "bert-base-uncased"
    logger.info(f"Falling back to: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForQuestionAnswering.from_pretrained(model_name)

# -----------------------------
# 6. Tokenization function (FIXED)
# -----------------------------
max_length = 384
doc_stride = 128

def preprocess_function(examples):
    """Tokenize examples for QA training"""
    
    # Handle single example vs batch
    if isinstance(examples["question"], str):
        questions = [examples["question"]]
        contexts = [examples["context"]]
        all_answers = [examples["answers"]]
    else:
        questions = examples["question"]
        contexts = examples["context"]
        all_answers = examples["answers"]
    
    tokenized = tokenizer(
        questions,
        contexts,
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(tokenized["offset_mapping"]):
        # Get original example index (in case of overflow)
        sample_index = tokenized["overflow_to_sample_mapping"][i] if "overflow_to_sample_mapping" in tokenized else i
        
        if sample_index < len(all_answers):
            answers = all_answers[sample_index]
            answer_starts = answers["answer_start"]
            answer_texts = answers["text"]
            
            # Use first answer only
            if answer_starts and answer_texts and answer_starts[0] != -1:
                start_char = answer_starts[0]
                answer_text = answer_texts[0]
                end_char = start_char + len(answer_text)
            else:
                # No valid answer
                start_positions.append(0)
                end_positions.append(0)
                continue
        else:
            # Overflow case with no answer
            start_positions.append(0)
            end_positions.append(0)
            continue

        sequence_ids = tokenized.sequence_ids(i)
        if sequence_ids is None:
            start_positions.append(0)
            end_positions.append(0)
            continue

        # Find context start and end in tokens
        context_start = 0
        context_end = len(sequence_ids) - 1
        
        # Find first context token (sequence_id = 1)
        while context_start < len(sequence_ids) and sequence_ids[context_start] != 1:
            context_start += 1
            
        # Find last context token
        while context_end >= 0 and sequence_ids[context_end] != 1:
            context_end -= 1

        # If answer is outside context, set to no-answer
        if (start_char < offsets[context_start][0] or 
            end_char > offsets[context_end][1]):
            start_positions.append(0)
            end_positions.append(0)
            continue

        # Find token positions
        token_start_index = context_start
        token_end_index = context_end
        
        # Find start token
        while (token_start_index <= context_end and 
               offsets[token_start_index][0] < start_char):
            token_start_index += 1
        
        # Find end token  
        while (token_end_index >= context_start and 
               offsets[token_end_index][1] > end_char):
            token_end_index -= 1
        
        start_positions.append(token_start_index)
        end_positions.append(token_end_index)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    
    # Remove offset mapping
    tokenized.pop("offset_mapping", None)
    if "overflow_to_sample_mapping" in tokenized:
        tokenized.pop("overflow_to_sample_mapping")
    
    return tokenized

# Apply tokenization
tokenized_train = train_dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train dataset"
)
tokenized_test = test_dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=test_dataset.column_names,
    desc="Tokenizing test dataset"
)

logger.info(f"Tokenized train samples: {len(tokenized_train)}")
logger.info(f"Tokenized test samples: {len(tokenized_test)}")

# -----------------------------
# 7. Improved metrics
# -----------------------------
def compute_metrics(eval_pred):
    """Compute QA metrics"""
    start_logits, end_logits = eval_pred.predictions
    start_labels, end_labels = eval_pred.label_ids
    
    start_predictions = np.argmax(start_logits, axis=1)
    end_predictions = np.argmax(end_logits, axis=1)
    
    # Exact match: both start and end positions must be correct
    exact_match = np.mean(
        (start_predictions == start_labels) & (end_predictions == end_labels)
    )
    
    # Individual accuracy
    start_accuracy = np.mean(start_predictions == start_labels)
    end_accuracy = np.mean(end_predictions == end_labels)
    
    return {
        "exact_match": exact_match,
        "start_accuracy": start_accuracy,
        "end_accuracy": end_accuracy
    }

# -----------------------------
# 8. Optimized Training arguments
# -----------------------------
from huggingface_hub import HfFolder

training_args = TrainingArguments(
    output_dir="/kaggle/working/modernbert-qa-kaggle",
    
    # Batch sizes - adjust based on GPU memory
    per_device_train_batch_size=16,  # Reduced from 32 for QA task
    per_device_eval_batch_size=8,
    
    # Learning rate and epochs
    learning_rate=5e-5,
    num_train_epochs=5,
    warmup_steps=500,
    weight_decay=0.01,
    
    # Optimizations
    bf16=True if torch.cuda.is_available() else False,  # bfloat16 training for better stability
    optim="adamw_torch_fused",  # improved optimizer for faster training
    dataloader_pin_memory=True,
    
    # Logging & evaluation strategies
    logging_strategy="steps",
    logging_steps=100,
    eval_strategy="epoch",  # Evaluate every epoch
    save_strategy="epoch",   # Save every epoch
    save_total_limit=2,      # Keep only 2 best checkpoints
    
    # Model selection
    load_best_model_at_end=True,
    metric_for_best_model="exact_match",
    greater_is_better=True,
    
    # Reporting and Hub upload
    report_to="tensorboard",  # Enable tensorboard logging
    push_to_hub=False,        # Set to True if you want to upload to HF Hub
    hub_strategy="every_save",
    # hub_token=HfFolder.get_token(),  # Uncomment if pushing to hub
    
    # Other settings
    seed=42,
    remove_unused_columns=True,
)

# -----------------------------
# 9. Initialize trainer with optimized settings
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# -----------------------------
# 10. Train model
# -----------------------------
try:
    logger.info("Starting training...")
    trainer.train()
    
    # Save final model (after training completes)
    final_model_path = "/kaggle/working/final_qa_model"
    trainer.save_model(final_model_path)
    tokenizer.save_pretrained(final_model_path)
    
    logger.info("Training completed successfully!")
    logger.info(f"Model saved to: {final_model_path}")
    logger.info(f"Checkpoints saved to: {training_args.output_dir}")
    
    # Optional: Push to Hugging Face Hub
    if training_args.push_to_hub:
        try:
            trainer.push_to_hub(commit_message="Fine-tuned QA model for legal contradiction detection")
            logger.info("Model pushed to Hugging Face Hub!")
        except Exception as e:
            logger.warning(f"Failed to push to hub: {e}")
    
    # Final evaluation
    eval_results = trainer.evaluate()
    logger.info(f"Final evaluation results: {eval_results}")
    
except Exception as e:
    logger.error(f"Training failed: {e}")
    raise

# -----------------------------
# 11. Inference function
# -----------------------------
def predict_contradiction_span(premise, hypothesis=None, model_path="/kaggle/working/final_qa_model"):
    """Predict contradiction spans for new examples"""
    try:
        # Load trained model
        trained_model = AutoModelForQuestionAnswering.from_pretrained(model_path)
        trained_tokenizer = AutoTokenizer.from_pretrained(model_path)
        
        question = "Find span in premise that contradicts hypothesis"
        context = premise
        
        # Tokenize
        inputs = trained_tokenizer(
            question, 
            context, 
            return_tensors="pt", 
            truncation=True, 
            max_length=384
        )
        
        # Predict
        with torch.no_grad():
            outputs = trained_model(**inputs)
        
        # Get predictions
        start_scores = outputs.start_logits[0]
        end_scores = outputs.end_logits[0]
        
        start_idx = torch.argmax(start_scores).item()
        end_idx = torch.argmax(end_scores).item()
        
        # Decode answer
        if start_idx <= end_idx and start_idx > 0:
            answer_tokens = inputs["input_ids"][0][start_idx:end_idx+1]
            answer = trained_tokenizer.decode(answer_tokens, skip_special_tokens=True)
            
            # Get confidence scores
            start_prob = torch.softmax(start_scores, dim=0)[start_idx].item()
            end_prob = torch.softmax(end_scores, dim=0)[end_idx].item()
            confidence = (start_prob + end_prob) / 2
            
            return {
                "answer": answer.strip(),
                "confidence": confidence,
                "start_pos": start_idx,
                "end_pos": end_idx
            }
        else:
            return {
                "answer": "No contradiction found",
                "confidence": 0.0,
                "start_pos": 0,
                "end_pos": 0
            }
            
    except Exception as e:
        logger.error(f"Prediction failed: {e}")
        return {"error": str(e)}

# Example usage after training
logger.info("\n" + "="*50)
logger.info("Training completed! You can now use the model for inference:")
logger.info("="*50)

# Uncomment to test after training:
# example_premise = "Workers are entitled to 30 days of paid vacation per year according to labor law."
# result = predict_contradiction_span(example_premise)
# logger.info(f"Example prediction: {result}")

## 4. Cấu hình Tham số Huấn luyện
Thiết lập Learning Rate, Batch Size và các chiến lược đánh giá.


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate


## 1. Nạp và Chuẩn bị Dữ liệu
Phân tách tập dữ liệu thành các tập Train, Validation và Test.


In [ ]:
import os
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import f1_score, accuracy_score
from transformers.trainer_utils import get_last_checkpoint


# ===========================
# 1. Load dataset JSONL
# ===========================
dataset = load_dataset(
    "json",
    data_files="/kaggle/input/doc-nli/docnli_final_merged.jsonl"
)


# ===========================
# 2. Chuẩn hóa nhãn (fix typo)
# ===========================
def fix_gold_label(example):
    label = example["gold_label"].strip().lower()
    if label == "contradition":
        example["gold_label"] = "contradiction"
    return example

dataset = dataset.map(fix_gold_label)


# ===========================
# 3. Shuffle & chia train/val/test
# ===========================
dataset = dataset["train"].shuffle(seed=42).train_test_split(test_size=0.2, seed=42)
train_valid = dataset["train"].train_test_split(test_size=0.1, seed=42)  # 10% validation

train_data = train_valid["train"]
valid_data = train_valid["test"]
test_data = dataset["test"]


# ===========================
# 4. Định nghĩa nhãn
# ===========================
label2id = {"entailment": 0, "neutral": 1, "contradiction": 2}
id2label = {v: k for k, v in label2id.items()}


# ===========================
# 5. Load Model + Tokenizer
# ===========================
model_name = "tasksource/modernbert-base-nli"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)


# ===========================
# 6. Tokenization
# ===========================
def preprocess(batch):
    encodings = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    encodings["labels"] = [label2id[label] for label in batch["gold_label"]]
    return encodings

train_dataset = train_data.map(preprocess, batched=True)
valid_dataset = valid_data.map(preprocess, batched=True)
test_dataset = test_data.map(preprocess, batched=True)


# ===========================
# 7. Metrics (Accuracy + F1 Macro)
# ===========================
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }


# ===========================
# 8. Training Args
# ===========================
training_args = TrainingArguments(
    output_dir="./crimebert-nli-checkpoints",
    eval_strategy="steps",   # đánh giá theo steps
    save_strategy="steps",         # lưu checkpoint theo steps
    save_steps=500,                # lưu mỗi 500 step
    eval_steps=500,                # đánh giá mỗi 500 step
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    logging_dir="./logs",
    report_to="none",
    save_total_limit=3,            # chỉ giữ lại 3 checkpoint gần nhất
    logging_steps=100,
    fp16=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True
)


# ===========================
# 9. Trainer
# ===========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


# ===========================
# 10. Train (resume nếu có checkpoint)
# ===========================
last_checkpoint = None
if os.path.isdir(training_args.output_dir):
    last_checkpoint = get_last_checkpoint(training_args.output_dir)

if last_checkpoint is not None:
    print(f"🔄 Resume training from checkpoint: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    trainer.train()


# ===========================
# 11. Evaluate
# ===========================
print("📊 Evaluate on Validation Set:")
valid_metrics = trainer.evaluate(valid_dataset)
print(valid_metrics)

print("📊 Evaluate on Test Set:")
test_metrics = trainer.evaluate(test_dataset)
print(test_metrics)


2025-09-26 01:22:23.478069: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758849743.500229     304 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758849743.507000     304 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

/tmp/ipykernel_304/1041400449.py:130: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss,Validation Loss,Accuracy,F1 Macro
500,0.017100,0.030208,0.996250,0.996267
1000,0.000000,0.031648,0.997500,0.997516


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


📊 Evaluate on Validation Set:


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


{'eval_loss': 0.03164830803871155, 'eval_accuracy': 0.9975, 'eval_f1_macro': 0.9975159133318613, 'eval_runtime': 9.035, 'eval_samples_per_second': 88.544, 'eval_steps_per_second': 1.439, 'epoch': 10.0}
📊 Evaluate on Test Set:


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


## 1. Nạp và Chuẩn bị Dữ liệu
Phân tách tập dữ liệu thành các tập Train, Validation và Test.


In [ ]:
import os
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

# 1. Load dataset JSON
dataset = load_dataset(
    "json",
    data_files="/kaggle/input/nli-dataset-coversation/merged_nli_dataset.json"
)

# 2. Sửa lỗi nhãn
def fix_gold_label(example):
    label = example["gold_label"].strip().lower()
    if label == "contradition":
        example["gold_label"] = "contradiction"
    return example

dataset = dataset.map(fix_gold_label)

# 3. Chia tập train/test
dataset = dataset["train"].train_test_split(test_size=0.2)
train_data = dataset["train"]
eval_data = dataset["test"]

# 4. Định nghĩa nhãn
label2id = {"entailment": 0, "neutral": 1, "contradiction": 2}
id2label = {v: k for k, v in label2id.items()}

# 5. Load tokenizer + model
model_name = "tasksource/ModernBERT-base-nli"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

# 6. Tokenize
def preprocess(batch):
    encodings = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    encodings["labels"] = [label2id[label] for label in batch["gold_label"]]
    return encodings

train_dataset = train_data.map(preprocess, batched=True)
eval_dataset = eval_data.map(preprocess, batched=True)

# 7. Metric
metric = evaluate.load("accuracy")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# ================================
# GRID SEARCH HYPERPARAMETERS
# ================================

learning_rates = [1e-5, 2e-5, 3e-5]
batch_sizes = [8, 16]

best_acc = 0
best_config = {}

for lr in learning_rates:
    for bs in batch_sizes:
        print(f"\n===== Training with LR={lr}, Batch Size={bs} =====")

        training_args = TrainingArguments(
            output_dir=f"./modernbert-nli-lr{lr}-bs{bs}",
            eval_strategy="epoch",
            save_strategy="epoch",
            learning_rate=lr,
            per_device_train_batch_size=bs,
            per_device_eval_batch_size=bs,
            num_train_epochs=5,              # train nhiều epoch nhưng sẽ dừng sớm
            weight_decay=0.01,
            load_best_model_at_end=True,
            logging_dir="./logs",
            report_to="none",
            save_total_limit=2
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            tokenizer=tokenizer,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]  # dừng nếu 2 lần val không cải thiện
        )

        trainer.train()

        metrics = trainer.evaluate()
        acc = metrics["eval_accuracy"]
        print(f"Accuracy: {acc}")

        if acc > best_acc:
            best_acc = acc
            best_config = {"lr": lr, "batch_size": bs}

print("\n==============================")
print(f"Best config: {best_config}, Accuracy={best_acc}")


## 6. Lưu trữ & Đóng gói Mô hình
Lưu model tốt nhất và đóng gói (zip) để sử dụng cho việc triển khai.


In [ ]:
# 14. Lưu model và tokenizer
trainer.save_model("./modernbert-nli-conv/best_model")
tokenizer.save_pretrained("./modernbert-nli-conv/best_model")
